# H5: Metacognitive Monitoring (Bayesian)

- H5a: Calibration → optimality (LOO comparison)
- H5b: Anxiety slope → choice shift
- H5c: ω → confidence, not anxiety
- H5d: Confidence → error type

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, os
import bambi as bmb, arviz as az
from scipy.stats import pearsonr, zscore, linregress
from config import *
from load_data import load_both

%matplotlib inline

exp_data, conf_data = load_both()
all_data = {'exploratory': exp_data, 'confirmatory': conf_data}
all_data = {k: v for k, v in all_data.items() if v is not None}

for name, d in all_data.items():
    print(f'{d["config"].label}: master={len(d["master"])} subjects')


## Load params + build master

In [ ]:
# Prepare z-scored affect indices for each sample
for name, d in all_data.items():
    m = d['master']
    # Z-score affect indices within-sample
    for col, zcol in [('anx_calibration', 'calibration_z'), ('anx_slope', 'anx_slope_z'),
                       ('mean_confidence', 'confidence_z'), ('mean_anxiety', 'anxiety_z')]:
        if col in m.columns:
            vals = m[col].dropna()
            m[zcol] = (m[col] - vals.mean()) / vals.std()
    d['master'] = m
    print(f'{d["config"].label}: master={len(m)} subjects, '
          f'has calibration={m["anx_calibration"].notna().sum()}, '
          f'has confidence={m["mean_confidence"].notna().sum()}')

## H5a: Calibration → optimality (LOO comparison)

In [ ]:
results_h5a = {}

for name, d in all_data.items():
    label = d['config'].label
    m = d['master']
    v = m.dropna(subset=['omega_z', 'kappa_z', 'calibration_z', 'pct_opt', 'escape_rate', 'earnings'])
    print(f'\n{"="*60}')
    print(f'H5a — Calibration beyond omega+kappa — {label} (N={len(v)})')
    print(f'{"="*60}')

    sample_results = {}
    for outcome, olabel in [('pct_opt', 'Optimality'), ('escape_rate', 'Escape'), ('earnings', 'Earnings')]:
        df = v[['omega_z', 'kappa_z', 'calibration_z', outcome]].dropna()
        m_base = bmb.Model(f'{outcome} ~ omega_z + kappa_z', data=df)
        r_base = m_base.fit(**BKW, idata_kwargs={"log_likelihood": True})
        m_full = bmb.Model(f'{outcome} ~ omega_z + kappa_z + calibration_z', data=df)
        r_full = m_full.fit(**BKW, idata_kwargs={"log_likelihood": True})

        loo_b = az.loo(r_base)
        loo_f = az.loo(r_full)
        delta = loo_f.elpd_loo - loo_b.elpd_loo

        s_f = az.summary(r_full, hdi_prob=0.95)
        lo = s_f.loc['calibration_z', 'hdi_2.5%']
        hi = s_f.loc['calibration_z', 'hdi_97.5%']
        passed = lo > 0 or hi < 0
        sample_results[outcome] = {
            'pass': passed, 'delta_elpd': delta,
            'beta': s_f.loc['calibration_z', 'mean'], 'lo': lo, 'hi': hi,
        }
        print(f'  {olabel}: ΔELPD={delta:+.1f}, cal β={s_f.loc["calibration_z","mean"]:+.4f} '
              f'[{lo:+.4f},{hi:+.4f}] {"PASS" if passed else "FAIL"}')

    n_pass = sum(r['pass'] for r in sample_results.values())
    print(f'\n  H5a ({name}): {"PASS" if n_pass >= 2 else "FAIL"} ({n_pass}/3)')
    results_h5a[name] = sample_results

## H5b: Anxiety slope → choice shift

In [ ]:
results_h5b = {}

for name, d in all_data.items():
    label = d['config'].label
    m = d['master']
    v = m.dropna(subset=['anx_slope_z', 'choice_shift'])
    print(f'\n{"="*60}')
    print(f'H5b — Anxiety slope -> choice shift — {label} (N={len(v)})')
    print(f'{"="*60}')

    mod = bmb.Model('choice_shift ~ anx_slope_z', data=v)
    res = mod.fit(**BKW)
    s = az.summary(res, hdi_prob=0.95)
    lo = s.loc['anx_slope_z', 'hdi_2.5%']
    hi = s.loc['anx_slope_z', 'hdi_97.5%']
    beta = s.loc['anx_slope_z', 'mean']
    passed = lo > 0
    results_h5b[name] = {'pass': passed, 'beta': beta, 'lo': lo, 'hi': hi}
    print(f'  slope -> shift: beta={beta:+.4f}, 95% HDI=[{lo:+.4f},{hi:+.4f}]')
    print(f'  H5b ({name}): {"PASS" if passed else "FAIL"}')

## H5c: ω → confidence, not anxiety

In [ ]:
results_h5c = {}

for name, d in all_data.items():
    label = d['config'].label
    m = d['master']
    v = m.dropna(subset=['omega_z', 'mean_anxiety', 'mean_confidence'])
    print(f'\n{"="*60}')
    print(f'H5c — omega -> confidence, not anxiety — {label} (N={len(v)})')
    print(f'{"="*60}')

    # omega -> confidence (HDI should exclude 0, negative)
    m_c = bmb.Model('mean_confidence ~ omega_z', data=v)
    r_c = m_c.fit(**BKW)
    s_c = az.summary(r_c, hdi_prob=0.95)
    lo_c = s_c.loc['omega_z', 'hdi_2.5%']
    hi_c = s_c.loc['omega_z', 'hdi_97.5%']
    beta_c = s_c.loc['omega_z', 'mean']
    pass_conf = hi_c < 0

    # omega -> anxiety (ROPE null: HDI should include 0)
    m_a = bmb.Model('mean_anxiety ~ omega_z', data=v)
    r_a = m_a.fit(**BKW)
    s_a = az.summary(r_a, hdi_prob=0.95)
    lo_a = s_a.loc['omega_z', 'hdi_2.5%']
    hi_a = s_a.loc['omega_z', 'hdi_97.5%']
    beta_a = s_a.loc['omega_z', 'mean']
    pass_anx = lo_a < 0 and hi_a > 0  # includes zero

    results_h5c[name] = {
        'pass_conf': pass_conf, 'beta_conf': beta_c, 'lo_conf': lo_c, 'hi_conf': hi_c,
        'pass_anx': pass_anx, 'beta_anx': beta_a, 'lo_anx': lo_a, 'hi_anx': hi_a,
    }
    print(f'  omega -> conf: beta={beta_c:+.3f} [{lo_c:+.3f},{hi_c:+.3f}] {"PASS" if pass_conf else "FAIL"}')
    print(f'  omega -> anx:  beta={beta_a:+.3f} [{lo_a:+.3f},{hi_a:+.3f}] {"PASS" if pass_anx else "FAIL"} (should include zero)')
    print(f'  H5c ({name}): {"PASS" if pass_conf and pass_anx else "FAIL"}')

## H5d: Confidence → error type

In [ ]:
results_h5d = {}

for name, d in all_data.items():
    label = d['config'].label
    m = d['master']
    v = m.dropna(subset=['confidence_z', 'n_oc', 'n_rk'])
    print(f'\n{"="*60}')
    print(f'H5d — Confidence -> error type — {label} (N={len(v)})')
    print(f'{"="*60}')

    # confidence -> overcautious errors (HDI < 0)
    m_oc = bmb.Model('n_oc ~ confidence_z', data=v)
    r_oc = m_oc.fit(**BKW)
    s_oc = az.summary(r_oc, hdi_prob=0.95)
    lo_oc = s_oc.loc['confidence_z', 'hdi_2.5%']
    hi_oc = s_oc.loc['confidence_z', 'hdi_97.5%']
    beta_oc = s_oc.loc['confidence_z', 'mean']
    pass_oc = hi_oc < 0

    # confidence -> reckless errors (HDI > 0)
    m_rk = bmb.Model('n_rk ~ confidence_z', data=v)
    r_rk = m_rk.fit(**BKW)
    s_rk = az.summary(r_rk, hdi_prob=0.95)
    lo_rk = s_rk.loc['confidence_z', 'hdi_2.5%']
    hi_rk = s_rk.loc['confidence_z', 'hdi_97.5%']
    beta_rk = s_rk.loc['confidence_z', 'mean']
    pass_rk = lo_rk > 0

    results_h5d[name] = {
        'pass_oc': pass_oc, 'beta_oc': beta_oc, 'lo_oc': lo_oc, 'hi_oc': hi_oc,
        'pass_rk': pass_rk, 'beta_rk': beta_rk, 'lo_rk': lo_rk, 'hi_rk': hi_rk,
    }
    print(f'  conf -> OC: beta={beta_oc:+.2f} [{lo_oc:+.2f},{hi_oc:+.2f}] {"PASS" if pass_oc else "FAIL"}')
    print(f'  conf -> rk: beta={beta_rk:+.2f} [{lo_rk:+.2f},{hi_rk:+.2f}] {"PASS" if pass_rk else "FAIL"}')
    print(f'  H5d ({name}): {"PASS" if pass_oc and pass_rk else "FAIL"}')

## Summary

In [ ]:
print('H5 SUMMARY (Bayesian) — Both Samples')
print('=' * 80)

# Build test definitions
test_defs = [
    ('H5a', 'Cal -> optimality',  lambda n: results_h5a[n]['pct_opt']['pass']),
    ('H5a', 'Cal -> escape',      lambda n: results_h5a[n]['escape_rate']['pass']),
    ('H5a', 'Cal -> earnings',    lambda n: results_h5a[n]['earnings']['pass']),
    ('H5b', 'Slope -> shift',     lambda n: results_h5b[n]['pass']),
    ('H5c', 'omega -> confidence', lambda n: results_h5c[n]['pass_conf']),
    ('H5c', 'omega -> anxiety null', lambda n: results_h5c[n]['pass_anx']),
    ('H5d', 'Conf -> OC',         lambda n: results_h5d[n]['pass_oc']),
    ('H5d', 'Conf -> reckless',   lambda n: results_h5d[n]['pass_rk']),
]

sample_names = list(all_data.keys())
header = f'  {"Hyp":<6} {"Test":<24}'
for sn in sample_names:
    header += f' {sn:<16}'
print(header)
print('-' * 80)

totals = {sn: 0 for sn in sample_names}
for hyp, tname, fn in test_defs:
    row = f'  {hyp:<6} {tname:<24}'
    for sn in sample_names:
        if sn in results_h5a:  # sample exists
            try:
                p = fn(sn)
                row += f' {"PASS" if p else "FAIL":<16}'
                if p:
                    totals[sn] += 1
            except KeyError:
                row += f' {"N/A":<16}'
        else:
            row += f' {"N/A":<16}'
    print(row)

print('-' * 80)
for sn in sample_names:
    print(f'  {sn}: {totals[sn]}/{len(test_defs)} tests pass')
print('=' * 80)